# S03 · Cleaning a messy Excel file

This is the lab. Real data arrives dirty: blanks written in odd ways, duplicate
rows, numbers stored as text, and the same name typed inconsistently. We read a
real, messy sales **workbook** that ships with this session — the kind a regional
manager would email you — and clean it end to end, exactly as the shop owner needs
before her month-end review.

**New here? Read this once.**

- New to Python? Run each cell top to bottom with the play button and read the
  note above it. The whole notebook runs on its own; the data file ships with the
  session, and if it is missing the notebook recreates it.
- This is the honest, unglamorous part of every data job: getting the mess into
  shape before you can trust a single number. It is worth doing slowly.
- Already comfortable? Look for the cell marked **Stretch (optional)** at the end,
  where we join a region lookup and hand a clean two-sheet workbook back.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. This notebook reads and writes real
Excel files, which pandas does through a helper library called **openpyxl**. Colab
usually has it already; the line below makes sure. On your **own machine** you
installed everything with `uv`, so this does nothing there.

In [1]:
# On Colab, make sure the Excel engine openpyxl is present.
# Off Colab (your machine, or the course environment) it is already installed,
# so this cell installs nothing.
import sys
import subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
print("Setup complete.")

Setup complete.


In [2]:
import pandas as pd
import numpy as np   # we use np.nan (the "blank" / missing marker) later

## Step 1 — read the messy workbook (every sheet at once)

The file `shop_sales_messy.xlsx` ships in this session's `data/` folder. It has
one sheet per region (North, South, West) plus a small `lookup` sheet. We read
**all** the sheets at once with `sheet_name=None`, which hands back a dictionary
of `{sheet name: table}`. We also tell pandas that the text markers `"NA"` and
`"-"` mean *missing*, so it blanks them for us.

In [3]:
from pathlib import Path

# Read the messy workbook three ways, so this notebook runs anywhere:
#   1) from the repo's data/ folder, 2) else the course website (Colab),
#   3) else rebuild it as a last resort.
xlsx_path = Path("../data/shop_sales_messy.xlsx")
DATA_URL = "https://girishmkulkarni.github.io/applied-maths-in-industry-site/sessions/S03/data/shop_sales_messy.xlsx"

def read_all_sheets(src):
    # read EVERY sheet at once -> a dict of {sheet name: table};
    # na_values lists the TEXT markers that mean 'missing'
    return pd.read_excel(src, sheet_name=None, na_values=["NA", "-"])

if xlsx_path.exists():
    book = read_all_sheets(xlsx_path)
else:
    try:
        book = read_all_sheets(DATA_URL)
    except Exception:
        # rebuild the workbook locally, then read it
        north = pd.DataFrame({
            "date": ["2024-02-01", "2024-02-03", "2024-02-03", "2024-02-05"],
            "city": ["Pune", "Pune", "Pune", "Mumbai"],
            "customer": ["Asha", "Meera", "Meera", "Ravi"],
            "units": [10, 7, 7, "NA"],
            "price": ["250", "250", "250", "1,300"]})
        south = pd.DataFrame({
            "date": ["2024-02-02", "2024-02-04", "2024-02-06"],
            "city": ["Delhi", "Delhi", " hyderabad"],
            "customer": ["John", "Vikram", "Latha"],
            "units": [2, "-", 999],
            "price": ["180", "180", "1,210"]})
        west = pd.DataFrame({
            "date": ["2024-02-07", "2024-02-08", "2024-02-09"],
            "city": ["Nagpur", "Nashik", "nagpur "],
            "customer": ["Kiran", "Sara", "Asha"],
            "units": [5, 8, 6],
            "price": ["220", "", "1,220"]})
        lookup = pd.DataFrame({
            "city": ["Pune", "Mumbai", "Delhi", "Hyderabad", "Nagpur", "Nashik"],
            "region": ["North", "North", "South", "South", "West", "West"]})
        with pd.ExcelWriter("shop_sales_messy.xlsx", engine="openpyxl") as _xl:
            north.to_excel(_xl, sheet_name="North", index=False)
            south.to_excel(_xl, sheet_name="South", index=False)
            west.to_excel(_xl, sheet_name="West", index=False)
            lookup.to_excel(_xl, sheet_name="lookup", index=False)
        book = read_all_sheets("shop_sales_messy.xlsx")

print("sheets in the file:", list(book.keys()))
print()
print("the North sheet as read back:")
print(book["North"])

sheets in the file: ['North', 'South', 'West', 'lookup']

the North sheet as read back:
         date    city customer  units  price
0  2024-02-01    Pune     Asha   10.0    250
1  2024-02-03    Pune    Meera    7.0    250
2  2024-02-03    Pune    Meera    7.0    250
3  2024-02-05  Mumbai     Ravi    NaN  1,300


## Step 2 — stack the regional sheets into one table

The three regional sheets have the same columns, so we glue them into one long
table with `pd.concat`. We keep the `lookup` sheet aside for the Stretch at the
end. `ignore_index=True` renumbers the rows 0, 1, 2, … after stacking.

In [4]:
regions = ["North", "South", "West"]
raw = pd.concat([book[name] for name in regions], ignore_index=True)

print("all regions stacked into one table:")
print(raw)

all regions stacked into one table:
         date        city customer  units  price
0  2024-02-01        Pune     Asha   10.0    250
1  2024-02-03        Pune    Meera    7.0    250
2  2024-02-03        Pune    Meera    7.0    250
3  2024-02-05      Mumbai     Ravi    NaN  1,300
4  2024-02-02       Delhi     John    2.0    180
5  2024-02-04       Delhi   Vikram    NaN    180
6  2024-02-06   hyderabad    Latha  999.0  1,210
7  2024-02-07      Nagpur    Kiran    5.0    220
8  2024-02-08      Nashik     Sara    8.0    NaN
9  2024-02-09     nagpur      Asha    6.0  1,220


## Step 3 — inspect the problems before touching anything

Before fixing anything, look at what is wrong. Three quick checks: how many
missing values sit in each column, how many duplicate rows there are, and the
type of each column — `price` came back as text (object) because of the comma
separators like `"1,300"`.

In [5]:
print("missing values per column:")
print(raw.isna().sum())

print()
print("number of duplicate rows:", raw.duplicated().sum())

print()
print("data type of each column:")
print(raw.dtypes)

missing values per column:
date        0
city        0
customer    0
units       2
price       1
dtype: int64

number of duplicate rows: 1

data type of each column:
date         object
city         object
customer     object
units       float64
price        object
dtype: object


## Step 4 — remove the duplicate row

`drop_duplicates` keeps the first copy of any repeated row and drops the rest. We
work on a copy called `clean` so the original `raw` is left untouched.

In [6]:
clean = raw.copy()
clean = clean.drop_duplicates()

print("after removing duplicates:")
print(clean)

after removing duplicates:
         date        city customer  units  price
0  2024-02-01        Pune     Asha   10.0    250
1  2024-02-03        Pune    Meera    7.0    250
3  2024-02-05      Mumbai     Ravi    NaN  1,300
4  2024-02-02       Delhi     John    2.0    180
5  2024-02-04       Delhi   Vikram    NaN    180
6  2024-02-06   hyderabad    Latha  999.0  1,210
7  2024-02-07      Nagpur    Kiran    5.0    220
8  2024-02-08      Nashik     Sara    8.0    NaN
9  2024-02-09     nagpur      Asha    6.0  1,220


## Step 5 — turn the numeric `999` code into a real blank

Reading the file handled the *text* markers (`"NA"`, `"-"`), but the number
`999` in the `units` column is a 'missing' code in disguise: it slipped through
because it looks like a real number. We replace every `999` with `np.nan`, the
proper marker for a missing value.

In [7]:
clean["units"] = clean["units"].replace(999, np.nan)

print("after replacing 999 with a blank:")
print(clean)

after replacing 999 with a blank:
         date        city customer  units  price
0  2024-02-01        Pune     Asha   10.0    250
1  2024-02-03        Pune    Meera    7.0    250
3  2024-02-05      Mumbai     Ravi    NaN  1,300
4  2024-02-02       Delhi     John    2.0    180
5  2024-02-04       Delhi   Vikram    NaN    180
6  2024-02-06   hyderabad    Latha    NaN  1,210
7  2024-02-07      Nagpur    Kiran    5.0    220
8  2024-02-08      Nashik     Sara    8.0    NaN
9  2024-02-09     nagpur      Asha    6.0  1,220


## Step 6 — fix the price column (text → numbers)

The `price` column came in as text, with comma separators like `"1,300"`, so we
could not do arithmetic on it. We strip the commas, then convert to real numbers
with `pd.to_numeric`. `errors="coerce"` turns anything that still isn't a number
(like the blank price) into `NaN`.

In [8]:
# strip the thousands-commas, then convert the text to real numbers
clean["price"] = clean["price"].astype(str).str.replace(",", "", regex=False)
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")

print("data type of each column now:")
print(clean.dtypes)

data type of each column now:
date         object
city         object
customer     object
units       float64
price       float64
dtype: object


## Step 7 — fill in the remaining gaps

Two common choices for a gap: drop the row, or fill it. Here we fill the missing
`units` and `price` with that column's average — a simple, common choice. Which
is right depends on *why* the value is missing; filling is fine here.

In [9]:
clean["units"] = clean["units"].fillna(clean["units"].mean())
clean["price"] = clean["price"].fillna(clean["price"].mean())

print("the cleaned table:")
print(clean)

print()
print("summary of the number columns:")
print(clean.describe())

the cleaned table:
         date        city customer      units    price
0  2024-02-01        Pune     Asha  10.000000   250.00
1  2024-02-03        Pune    Meera   7.000000   250.00
3  2024-02-05      Mumbai     Ravi   6.333333  1300.00
4  2024-02-02       Delhi     John   2.000000   180.00
5  2024-02-04       Delhi   Vikram   6.333333   180.00
6  2024-02-06   hyderabad    Latha   6.333333  1210.00
7  2024-02-07      Nagpur    Kiran   5.000000   220.00
8  2024-02-08      Nashik     Sara   8.000000   601.25
9  2024-02-09     nagpur      Asha   6.000000  1220.00

summary of the number columns:
           units        price
count   9.000000     9.000000
mean    6.333333   601.250000
std     2.160247   498.583932
min     2.000000   180.000000
25%     6.000000   220.000000
50%     6.333333   250.000000
75%     7.000000  1210.000000
max    10.000000  1300.000000


## Step 8 — clean the city text so the join will match

The `city` column arrived with inconsistent text: `" hyderabad"` with a leading
space and lowercase, `"nagpur "` with a trailing space. To a computer these are
different from the clean `"Hyderabad"` and `"Nagpur"` in the lookup sheet, so the
join later would silently miss those rows. We strip the spaces and Title-Case the
names first — **clean your keys before you merge.**

In [10]:
# the city column has messy text that will NOT match the lookup sheet
print("cities before cleaning:", sorted(clean["city"].unique()))

# .str tools clean a whole text column at once
clean["city"] = clean["city"].str.strip().str.title()

print("cities after cleaning :", sorted(clean["city"].unique()))

cities before cleaning: [' hyderabad', 'Delhi', 'Mumbai', 'Nagpur', 'Nashik', 'Pune', 'nagpur ']
cities after cleaning : ['Delhi', 'Hyderabad', 'Mumbai', 'Nagpur', 'Nashik', 'Pune']


## Step 9 — label orders with `.loc`

A quick derived label: mark each order `big` or `small`. The right way to change
a chosen set of rows is `df.loc[rows, column] = value` — it edits the table in
place. Avoid `df[mask]["col"] = value`, which often changes a throwaway copy and
prints a `SettingWithCopyWarning`.

In [11]:
# label every order "big", then overwrite ONLY the small ones -- in one step
clean["order_size"] = "big"
clean.loc[clean["units"] < 5, "order_size"] = "small"

print(clean[["city", "customer", "units", "order_size"]])

        city customer      units order_size
0       Pune     Asha  10.000000        big
1       Pune    Meera   7.000000        big
3     Mumbai     Ravi   6.333333        big
4      Delhi     John   2.000000      small
5      Delhi   Vikram   6.333333        big
6  Hyderabad    Latha   6.333333        big
7     Nagpur    Kiran   5.000000        big
8     Nashik     Sara   8.000000        big
9     Nagpur     Asha   6.000000        big


### Stretch (optional) — join the region lookup, hand back a clean workbook

Skip this if you have had enough for one notebook. If you want the move that
makes you genuinely useful in an office, here it is. Because we cleaned the city
names in Step 8, the `lookup` sheet now matches every row, so the join attaches
each city's region cleanly. Then we write the tidy data and a small per-region
summary into a single Excel file with two sheets — the finance team can open it
with no Python at all.

In [12]:
# attach each city's region from the lookup sheet (a left join)
clean = pd.merge(clean, book["lookup"], on="city", how="left")

# a small per-region summary: total units and total value
clean["value"] = clean["units"] * clean["price"]
per_region = clean.groupby("region")[["units", "value"]].sum()

# write two sheets into one workbook: the tidy data, and the summary
output_filename = "clean_sales.xlsx"
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    clean.to_excel(writer, sheet_name="clean_data", index=False)
    per_region.to_excel(writer, sheet_name="summary_by_region")

print("Wrote a clean two-sheet workbook to", output_filename)
print()
print("the per-region summary sheet:")
print(per_region)

Wrote a clean two-sheet workbook to clean_sales.xlsx

the per-region summary sheet:
            units         value
region                         
North   23.333333  12483.333333
South   14.666667   9163.333333
West    19.000000  13230.000000


## What you just did

You read a real, messy multi-sheet Excel workbook, stacked the sheets into one
table, and cleaned it end to end: caught the text missing-markers, removed a
duplicate row, turned the numeric `999` code into a real blank, fixed a price
column stored as text with commas, filled the gaps, cleaned inconsistent city
text so a join would match, labelled rows with `.loc`, joined a region lookup, and
wrote a clean two-sheet workbook back. That loop — messy in, tidy out — is most of
the work in any real data project, and exactly what the shop owner needed.